## Early Transformer history

Here are some reference points at the beginning of the (short) history of Transformer models:

The [Transformer architecture](https://arxiv.org/abs/1706.03762) was introduced in June 2017. The focus of the original research was on `translation` tasks. This was followed by the introduction of several influential models, including:

- **June 2018**: [GPT](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf), the first pretrained Transformer model, used for fine-tuning on various NLP tasks and obtained state-of-the-art results

- **October 2018**: [BERT](https://arxiv.org/abs/1810.04805), another large pretrained model, this one designed to produce better summaries of sentences.

- **February 2019**: [GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf), an improved (and bigger) version of GPT that was not immediately publicly released due to ethical concerns

- **October 2019**: [T5](https://huggingface.co/papers/1910.10683), A multi-task focused implementation of the sequence-to-sequence Transformer architecture.

Those selected example can be grouped into three categories:

- GPT-like (also called _auto-regressive_ Transformer models)
- BERT-like (also called _auto-encoding_ Transformer models) 
- T5-like (also called _sequence-to-sequence_ Transformer models)

## General Transformer architecture

In this section, we'll go over the general architecture of the Transformer model. Don't worry if you don't understand some of the concepts; there are detailed sections later covering each of the components.

The model is primarily composed of two blocks:

* **Encoder (left)**: The encoder receives an input and builds a representation of it (its features). This means that the model is optimized to acquire understanding from the input.
* **Decoder (right)**: The decoder uses the encoder's representation (features) along with other inputs to generate a target sequence. This means that the model is optimized for generating outputs.

Each of these parts can be used independently, depending on the task: 

* **Encoder-only models**: Good for tasks that require understanding of the input, such as sentence classification and named entity recognition.
* **Decoder-only models**: Good for generative tasks such as text generation.
* **Encoder-decoder models** or **sequence-to-sequence models**: Good for generative tasks that require an input, such as translation or summarization.

## The original architecture

The Transformer architecture was originally designed for translation. During training, the encoder receives inputs (sentences) in a certain language, while the decoder receives the same sentences in the desired target language. In the encoder, the attention layers can use all the words in a sentence (since, as we just saw, the translation of a given word can be dependent on what is after as well as before it in the sentence). The decoder, however, works sequentially and can only pay attention to the words in the sentence that it has already translated (so, only the words before the word currently being generated). For example, when we have predicted the first three words of the translated target, we give them to the decoder  which then uses all the inputs of the encoder to try to predict the fourth word.

To speed things up during training (when the model has access to target sentences), the decoder is fed the whole target, but it is not allowed to use future words (if it had access to the word at position 2 when trying to predict the word at position 2, the problem would not be very hard!). For instance, when trying to predict the fourth word, the attention layer will only have access to the words in positions 1 to 3.

The original Transformer architecture looked like this, with the encoder on the left and the decoder on the right:

![image.png](./4341140b_image.png)

Note that the first attention layer in a decoder block pays attention to all (past) inputs to the decoder, but the second attention layer uses the output of the encoder. It can thus access the whole input sentence to best predict the current word. This is very useful as different languages can have grammatical rules that put the words in different orders, or some context provided later in the sentence may be helpful to determine the best translation of a given word.

The *attention mask* can also be used in the encoder/decoder to prevent the model from paying attention to some special words -- for instance, the special padding word used to make all the inputs the same length when batching together sentences.


### Types of language models

#### Main approaches

Language models work by being trained to predict the probability of a word given the context of surrounding words. This gives them a foundational understanding of language that can generalize to other tasks.

There are two main approaches for training a transformer model:

1. **Masked language modeling (MLM)**: Used by encoder models like BERT, this approach randomly masks some tokens in the input and trains the model to predict the original tokens based on the surrounding context. This allows the model to learn `bidirectional context` (looking at words both before and after the masked word) => _**No masked attention!**_ , the model can attend to all tokens in the sequence.

2. **Causal language modeling (CLM)**: Used by decoder models like GPT, this approach predicts the next token based on all previous tokens in the sequence. The model can only use context from the left (previous tokens) to predict the next token => _**Masked attention!**_ to prevent the model from seeing future tokens.

#### Architectural categories

Language models generally fall into three architectural categories:

1. **Encoder-only models** (like BERT): These models use a bidirectional approach to understand context from both directions. They're best suited for tasks that require deep understanding of text (of the full sequence), such as classification, named entity recognition, and question answering.

Encoder models use only the encoder of a Transformer model. At each stage, the attention layers can access all the words in the initial sentence. These models are often characterized as having “bi-directional” attention, and are often called auto-encoding models.

The pretraining of these models usually revolves around somehow corrupting a given sentence (for instance, by masking random words in it) and tasking the model with finding or reconstructing the initial sentence.

2. **Decoder-only models** (like GPT, Llama): These models process text from left to right and are particularly good at text generation tasks. They can complete sentences, write essays, or even generate code based on a prompt.

Decoder models use only the decoder of a Transformer model. At each stage, for a given word the attention layers can only access the words positioned before it in the sentence. These models are often called auto-regressive models.

The pretraining of decoder models usually revolves around predicting the next word in the sentence.

3. **Encoder-decoder models** (like T5, BART): These models combine both approaches, using an encoder to understand the input and a decoder to generate output. They excel at sequence-to-sequence tasks like translation, summarization, and question answering.

Encoder-decoder models (also called sequence-to-sequence models) use both parts of the Transformer architecture. At each stage, the attention layers of the encoder can access all the words in the initial sentence, whereas the attention layers of the decoder can only access the words positioned before a given word in the input.

The pretraining of these models can take different forms, but it often involves reconstructing a sentence for which the input has been somehow corrupted (for instance by masking random words). The pretraining of the T5 model consists of replacing random spans of text (that can contain several words) with a single mask special token (not only a single [MASK]), and the task is then to predict the text that this mask token replaces.

---

![transformer-models-for-language](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter1/transformers_architecture.png)

---

### Choosing the right architecture 

When working on a specific NLP task, how do you decide which architecture to use? Here's a quick guide:

| Task | Suggested Architecture | Examples |
|------|------------------------|----------|
| Text classification (sentiment, topic) | Encoder | BERT, RoBERTa |
| Text generation (creative writing) | Decoder | GPT, LLaMA |
| Translation | Encoder-Decoder | T5, BART |
| Summarization | Encoder-Decoder | T5, BART |
| Named entity recognition | Encoder | BERT, RoBERTa |
| Question answering (extractive) | Encoder | BERT, RoBERTa |
| Question answering (generative) | Encoder-Decoder or Decoder | T5, GPT |
| Conversational AI | Decoder | GPT, LLaMA |






In [1]:
from transformers import AutoModel
import io
from itertools import zip_longest

_models = ["google-bert/bert-base-uncased", "Qwen/Qwen3-0.6B", "facebook/bart-large-mnli"]  # N models

# 1. Load models and capture printed representation
model_strs = {}
for model_name in _models:
    model = AutoModel.from_pretrained(model_name)
    buf = io.StringIO()
    print(model, file=buf)
    model_strs[model_name] = buf.getvalue().strip().split("\n")  # list of lines
    del model

# 2. Compute max width for each model column
col_widths = []
for model_name in _models:
    lines = model_strs[model_name]
    max_len = max(len(l) for l in lines)
    col_widths.append(max_len + 4)  # padding

# 3. Print header row
header_cells = [
    model_name.ljust(col_widths[i])
    for i, model_name in enumerate(_models)
]
print(" | ".join(header_cells))

# 4. Print separator row
sep_cells = [
    "-" * col_widths[i]
    for i in range(len(_models))
]
print(" | ".join(sep_cells))

# 5. Print all rows side-by-side
# zip_longest creates rows of N columns
for row in zip_longest(*model_strs.values(), fillvalue=""):
    row_cells = [
        row[i].ljust(col_widths[i])
        for i in range(len(_models))
    ]
    print(" | ".join(row_cells))

google-bert/bert-base-uncased                                                      | Qwen/Qwen3-0.6B                                                                  | facebook/bart-large-mnli                                                                     
---------------------------------------------------------------------------------- | -------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------
BertModel(                                                                         | Qwen3Model(                                                                      | BartModel(                                                                                   
  (embeddings): BertEmbeddings(                                                    |   (embed_tokens): Embedding(151936, 1024)                                        |   (shared): BartScaledWordEmbedding(50265, 102

### So WHY Decoder-Only Won in modern LLMs?

The shift to decoder-only wasn't because encoder-decoder couldn't work—it was largely about **simplicity, scalability, and what worked best at scale**:

**Unification and Simplicity**: Decoder-only models can handle both understanding and generation with a single architecture. You can frame essentially any NLP task as text generation: translation becomes "Translate to French: [text]", classification becomes "Is this positive or negative?", etc. This unified framing meant one architecture, one training objective (next-token prediction), and simpler scaling.

**Scaling Efficiency**: When you're training trillion-parameter models on vast amounts of text, architectural simplicity matters enormously. Decoder-only models have a straightforward training loop—just predict the next token—which made it easier to scale to massive sizes. The encoder-decoder cross-attention adds complexity and computational overhead.

**Emergent Capabilities**: As decoder-only models scaled (GPT-2 → GPT-3 → beyond), they showed surprising emergent abilities through prompting alone. You didn't need the encoder-decoder structure because the decoder itself learned to "encode" context in its representations while generating.

**Pretraining Data Efficiency**: Decoder-only models can use all web text naturally—just predict what comes next. Encoder-decoder models often trained on specific paired tasks (translation, summarization), which was more data-constrained.

### ...LLMs can do so many tasks

> As we covered, language models are typically `pretrained` on large amounts of text data in a self-supervised manner (without human annotations), then `fine-tuned` on specific tasks. This approach, known as transfer learning, allows these models to adapt to many different NLP tasks with relatively small amounts of task-specific data.

The reason a generative LLM (typically a decoder-only model) is able to perform sequence-to-sequence tasks like **translation** and **summarization** is primarily due to two factors: **massive scale** and the use of the task as a **text generation** or **text continuation** problem.

---

### 💡 The Core Mechanism: Text Continuation

Decoder-only LLMs are fundamentally designed for **autoregression**, which means they are trained to predict the next token in a sequence based on all the preceding tokens. They treat the input prompt, the instruction, and the generated output as a **single, continuous stream of text**.

#### 1. Translation as Text Continuation

Instead of having a dedicated encoder to understand the source language and a decoder to generate the target language, a decoder-only model handles both within its single decoder stack:

* **Prompt Formatting:** The task is framed as a text generation prompt, for example: `"Translate the following English sentence to French: 'The cat sat on the mat.' French translation: "`
* **Sequential Processing:** The model processes the entire prompt up to the final colon. The input sentence, though written by the user, acts as the contextual information for the next-token prediction.
* **Output Generation:** The model then autoregressively generates the continuation—the French translation—token by token. The **self-attention** mechanism within the decoder allows it to look back at **every** part of the input sentence (the 'context') to inform the generation of the next word in the translation. This effectively allows the model to "encode" the input and "decode" the output within the same structure.

#### 2. Summarization as Text Continuation

The process for summarization is similar:

* **Prompt Formatting:** The instruction is appended to the full text: `"Summarize the following document in one paragraph: [Full Document Text] Summary: "`
* **Understanding and Compression:** The sheer size and training data of the LLM mean that its internal representations have learned to highly compress and understand the context of the long input text.
* **Output Generation:** It then generates a concise summary as the logical continuation of the prompt.

---

### 🚀 The Role of Scale and Data

While encoder-decoder models (like T5 and BART) are architecturally optimized for sequence-to-sequence tasks, the impressive performance of decoder-only LLMs (like GPT and LLaMA) comes down to their enormous size and training:

* **Massive Training Data:** Decoder-only models are trained on vast amounts of unlabelled text from the internet, which implicitly contains many examples of translation and summarization (e.g., news articles with headlines, paragraphs followed by their rephrased versions, parallel texts).
* **Emergent Abilities:** The scale of the models (billions/trillions of parameters) results in **emergent abilities**. They are not explicitly trained on a dedicated sequence-to-sequence objective, but the general skill of highly contextual next-token prediction generalizes to complex tasks like translation, summarization, and even coding, without needing a separate encoder component.

In essence, modern decoder-only LLMs leverage their exceptional ability to model and continue text to perform tasks that were once exclusively the domain of more specialized, two-part (encoder-decoder) architectures.

---

#### From first GPT paper (2018)

![image.png](./1b3f6d48_image.png)

[📘GPT paper](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)



